In [106]:
import random
import numpy as np
from scipy.signal import freqz

We begin by defining a function to generate an individual using **Ternary Encoding** or **Mixed Encoding**.

In [107]:
def get_coefficient(n_digits):
    trits = list()
    d_count = 0
    while d_count < n_digits:
        trit = random.choice([1, 0, -1]) 
        if trit != 0:
            d_count += 1
        trits.append(trit)
    return trits


def get_individual(order, n_digits):
    individual = list()
    for _ in range(order):
        individual.append(get_coefficient(n_digits))
    return individual


We also write a function to check the CSD correctness for both encodings.

In [108]:
def CSD_check(individual, n_digits):
    for coeff in individual:
        digit_count = 0
        for digit in coeff:
            if digit != 0:
                digit_count += 1
        if digit_count > n_digits:
            return False
    for coeff in individual:
        for i in range(len(coeff)-1):
            if coeff[i] != 0 and coeff[i+1] != 0:
                return False
    return True
    

We also write a utility to convert from the ternary encoding to a numeric coefficient.

In [109]:
def real_coefficient(coefficient):
    c = 0
    for i in range(len(coefficient)):
        c += coefficient[i] * (2 ** -i)
    return c

We can now write a population initialization function.

In [110]:
def init_population(pop_size, order, n_digits):
    pop = list()
    while len(pop) < pop_size:
        individual = get_individual(order, n_digits)
        #if CSD_check(individual, n_digits):
        pop.append(individual)
    return pop

To perform the mutation, we randomly select a coefficient, and change every trit according to a certain threshold.

In [111]:
def mut(individual):
    p = 0.5
    i = random.randint(0, len(individual)-1)
    for k in range(len(individual[i])):
        if random.random() < p:
            individual[i][k] = random.choice([1, 0, -1])
    return individual

For crossover, we perform a single point crossover between coefficients

In [112]:
def cross(parent1, parent2):
    i = random.randint(1, len(parent1)-1)
    child = parent1[:i] + parent2[i:]
    return child

The fitness function will return the inverse of the minimax error between the individual and the target frequency response. We will use scipy to model a FIR filter after the obtained coefficients

In [113]:
def minimax_error(individual, target):
    wi, Hi = freqz(individual)
    wt, Ht = freqz(target) 
    error = max(abs(abs(Ht) - abs(Hi)))
    return error

def fitness(individual, target):
    real_individual = [real_coefficient(coef) for coef in individual]
    error = minimax_error(real_individual, target)
    fitness = 1 / error
    return fitness

For selection, we use roulette wheel parent selection.

In [114]:
def roulette_selection(pop, pop_size, target):
    newpop = list()
    while len(newpop) < pop_size:
        total_fitness = sum([fitness(ind, target) for ind in pop])
        pick = random.uniform(0, total_fitness)
        current = 0
        for ind in pop:
            current += fitness(ind, target)
            if current > pick:
                newpop.append(ind)
                break
    return newpop

Now we have all the pieces to implement a simple version of GA.

In [115]:
# Parameters
pop_size = 10
order = 4  # Number of coefficients
n_digits = 3  # Number of non-zero digits per coefficient
target = [0.5, -0.46, 0.2, -0.7]  # Example target coefficients
generations = 100

# Initialize population
population = init_population(pop_size, order, n_digits)

for _ in range(generations):
    print(len(population))
    best_individual = max(population, key=lambda ind: fitness(ind, target))
    print("Best Individual:", [real_coefficient(coef) for coef in best_individual])
    print("Target: ", target)
    # Selection
    population = roulette_selection(population, pop_size,  target)
    
    # Crossover and Mutation
    next_generation = list()
    for i in range(0, len(population)-1, 2):
        parent1 = population[i]
        parent2 = population[i+1]
        child1 = cross(parent1, parent2)
        child2 = cross(parent2, parent1)
        next_generation.append(mut(child1))
        next_generation.append(mut(child2))
        next_generation.append(parent1)
        next_generation.append(parent2)
    population = next_generation

# Solution
best_individual = max(population, key=lambda ind: fitness(ind, target))
print("Best Individual:", [real_coefficient(coef) for coef in best_individual])

10
Best Individual: [-0.75, 0.75, -0.375, 0.515625]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [-0.75, 0.75, -0.375, 0.515625]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [-0.75, 1.25, 0.15625, 0.890625]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [0.5, 0.0, 0.15625, -0.609375]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [1.171875, 0.0, 0.125, -0.609375]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [-0.75, 0.25, 0.125, -0.375]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [1.28125, -1.0, 0.125, -0.609375]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [0.546875, -1.0, -0.875, 0.546875]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [-0.453125, -0.25, -0.875, 0.25]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [0.28125, -0.25, 0.9375, 0.25]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [1.40625, -1.25, -1.0625, -0.125]
Target:  [0.5, -0.46, 0.2, -0.7]
20
Best Individual: [-0.71875, 0.0, 0.1875, -0.25]
Target: 